In [1]:
import os
import sys
import torch
import tiktoken

project_root = os.path.dirname(os.path.abspath("")) # since notebook is in evaluation/
sys.path.insert(0, project_root)

import model
from model.model import GPT, GPTConfig
model.GPTConfig = GPTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"Using device: {device}")

Using device: mps


In [3]:
# Load configuration matching the notebook setup
config = GPTConfig(
    block_size=256,
    vocab_size=50257,
    n_layer=4,
    n_head=4,
    n_embd=128,
    dropout=0.1
)

# Initialize model
model = GPT(config)
model.to(device)
model.eval()

model_path = os.path.join(project_root, "training", "nanogpt_checkpoint_4.pt")
if not os.path.exists(model_path):
    print(f"Error: {model_path} not found.")
    print("Please run the notebook 'training/training_pipeline.ipynb' to train and save the model.")
else:
    # Load weights
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    if "model" in checkpoint:
        model.load_state_dict(checkpoint["model"])
    else:
        model.load_state_dict(checkpoint)
    print(f"Loaded model from {model_path}")

number of parameters: 7.23M
Loaded model from /Users/idant/Developer/Projects/NanoGPT/training/nanogpt_checkpoint_4.pt


In [4]:
# Setup tokenizer
enc = tiktoken.get_encoding("gpt2")

prompt = "[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]\nI ain't a killer but don't push me"
print(f"\nPrompt: '{prompt}'\n")

idx = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
max_new_tokens = 500


Prompt: '[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]
I ain't a killer but don't push me'



In [5]:
print("Temperature Sampling")
# Setting manual seed for determinism in generation examples
torch.manual_seed(42)
out_idx = model.generate(idx, max_new_tokens, temperature=0.7, repetition_penalty=1.2)
print(enc.decode(out_idx[0].tolist()))

Temperature Sampling
[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]
I ain't a killer but don't push me, yeah (Yeah)
They tryna make my style and you know I'ma let it rain (Yeah!)
I swear I was a man that we spent his momma as fuck like "Jatthean'"
For the homie, you're sittin', catch your face in off
And all the fuckin' niggas out of my bitches sayin' a nigga
Put me on, call him up, hit him up, watch him out
You was playin' off the trap up when he got with a tutor
He said she be a dirty, tell her quick for so
She was fucked 'em meanin' to have a motherfucker
Sorry they wishin' too many problems that?
Really just had a few lotta goin' off, bitch
Always wanted to gettin' ass in a baby
No one-money-got nigga say how this shit came
Or maybe I got the same rap game, it's all the same
So what it make me big dick
We gon' lose it, never leave it up
Now it over the street, no way you gotta let me down
It's nothin' gonna see alive
A lot of these hoes and the way